# Rare Disease, Real Kid: MVA Hackathon 2026 — exploración de datos

Dataset: [`SageBio/mva-hackathon-2026-data`](https://huggingface.co/datasets/SageBio/mva-hackathon-2026-data) (CC-BY-4.0, acceso gateado).

Contexto clínico: el paciente tiene sospecha/diagnóstico de **aneuploidía variegada mosaico (MVA)**, un síndrome de inestabilidad cromosómica asociado típicamente a mutaciones bialélicas en `BUB1B` (MVA1), `CEP57` (MVA2) o `TRIP13` (MVA3), con microcefalia, retraso de crecimiento, malformación de Dandy-Walker y predisposición a cáncer infantil.

**Nota ética/legal:** este dataset está detrás de un acuerdo de uso de datos (institución, compromiso de no recontactar a la familia, borrar los datos al terminar el hackathon, etc.). Ese acuerdo debe aceptarlo la persona dueña de la cuenta de Hugging Face en https://huggingface.co/datasets/SageBio/mva-hackathon-2026-data — este notebook asume que ya se hizo y que hay una sesión válida (`hf auth login` o variable de entorno `HF_TOKEN`).

In [ ]:
import gzip
import io
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from huggingface_hub import HfApi, hf_hub_download

REPO_ID = "SageBio/mva-hackathon-2026-data"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

VCF_NAME = "WGS_EX2312012_HGWCNDSX7.vcf.gz"
TBI_NAME = VCF_NAME + ".tbi"
DOCX_NAME = "Challenge_Clinical_Phenotype_1.docx"

## 1. Listado de archivos y tamaños

Esto funciona sin autenticación: los metadatos del repo son públicos aunque los archivos estén gateados. Los FASTQ (~85 GB en total) son lecturas crudas; para explorar variantes solo necesitamos el VCF ya llamado (~315 MB) y el documento de fenotipo clínico.

In [ ]:
api = HfApi()
info = api.repo_info(REPO_ID, repo_type="dataset", files_metadata=True)

rows = [(s.rfilename, (s.size or 0) / 1e9) for s in info.siblings]
df_files = pd.DataFrame(rows, columns=["archivo", "GB"]).sort_values("GB", ascending=False)
df_files

## 2. Descarga de los archivos gateados

Si esta celda falla con `GatedRepoError`, significa que la cuenta de Hugging Face activa (`hf auth whoami`) todavía no aceptó los términos del hackathon en la página del dataset.

In [ ]:
for fname in [DOCX_NAME, VCF_NAME, TBI_NAME]:
    path = hf_hub_download(REPO_ID, fname, repo_type="dataset", local_dir=str(DATA_DIR))
    print("OK:", path)

## 3. Fenotipo clínico

Texto completo del documento que describe los síntomas del paciente (útil para cruzar con genes candidatos más abajo).

In [ ]:
import docx

doc = docx.Document(DATA_DIR / DOCX_NAME)
for p in doc.paragraphs:
    if p.text.strip():
        print(p.text)

for table in doc.tables:
    for row in table.rows:
        print(" | ".join(cell.text.strip() for cell in row.cells))

## 4. Encabezado del VCF

Antes de tocar las variantes conviene ver el header: build del genoma de referencia, caller usado, y si ya viene anotado (campos `ANN=` de SnpEff o `CSQ=` de VEP en el INFO).

In [ ]:
meta_lines = []
header_line = None

with gzip.open(DATA_DIR / VCF_NAME, "rt") as f:
    for line in f:
        if line.startswith("##"):
            meta_lines.append(line.rstrip())
        elif line.startswith("#CHROM"):
            header_line = line.rstrip()
            break

print(f"{len(meta_lines)} líneas de metadatos\n")

interesting = [l for l in meta_lines if re.search(r"reference|source|GATKCommandLine|VEP|SnpEff|contig=<ID=chr?1,", l, re.IGNORECASE)]
for l in interesting[:20]:
    print(l)

print("\nColumnas:", header_line)

## 5. Cargar variantes a un DataFrame

Es un VCF de una sola muestra (~300 MB comprimido). Lo leemos directo con pandas usando el header real detectado arriba.

In [ ]:
colnames = header_line.lstrip("#").split("\t")

with gzip.open(DATA_DIR / VCF_NAME, "rt") as f:
    variants = pd.read_csv(f, sep="\t", comment="#", names=colnames, low_memory=False)

print(variants.shape)
variants.head()

## 6. QC básico

In [ ]:
print("Total de variantes:", len(variants))
print("\nPor FILTER:")
print(variants["FILTER"].value_counts())

is_snv = (variants["REF"].str.len() == 1) & (variants["ALT"].str.len() == 1)
print(f"\nSNVs: {is_snv.sum()}  |  indels/otros: {(~is_snv).sum()}")

transitions = {("A", "G"), ("G", "A"), ("C", "T"), ("T", "C")}
snv = variants[is_snv]
is_ts = list(zip(snv["REF"], snv["ALT"]))
n_ts = sum(1 for pair in is_ts if pair in transitions)
n_tv = len(is_ts) - n_ts
print(f"\nTi/Tv ratio: {n_ts / n_tv:.3f}  (esperado ~2.0-2.1 en WGS humano; sirve como check de calidad)")

## 7. Variantes por cromosoma

In [ ]:
counts = variants["CHROM"].value_counts()
order = [c for c in [f"chr{i}" for i in list(range(1, 23)) + ["X", "Y", "M"]] if c in counts.index] or counts.index
counts = counts.reindex(order).dropna()

fig, ax = plt.subplots(figsize=(12, 4))
counts.plot(kind="bar", ax=ax)
ax.set_ylabel("# variantes")
ax.set_title("Variantes por cromosoma")
plt.tight_layout()
plt.show()

## 8. Genes candidatos conocidos de MVA

`BUB1B` (MVA1, 15q15.1), `CEP57` (MVA2, 11q21), `TRIP13` (MVA3, 5p15.33). Coordenadas en **GRCh38** — si el header del VCF (paso 4) indica GRCh37/hg19, hay que usar las coordenadas equivalentes en esa build antes de filtrar.

Si el VCF no viene anotado con símbolo de gen (sin `ANN=`/`CSQ=` en INFO), filtramos por posición genómica directamente.

In [ ]:
# Coordenadas GRCh38 (ajustar si el VCF está en GRCh37/hg19)
candidate_genes = {
    "BUB1B": ("chr15", 40_449_029, 40_517_137),
    "CEP57": ("chr11", 95_384_873, 95_432_575),
    "TRIP13": ("chr5", 892_919, 913_045),
}

frames = []
for gene, (chrom, start, end) in candidate_genes.items():
    mask = (variants["CHROM"] == chrom) & variants["POS"].between(start, end)
    sub = variants.loc[mask].copy()
    sub["gene"] = gene
    frames.append(sub)

candidates = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Variantes dentro de genes candidatos: {len(candidates)}")
candidates[["gene", "CHROM", "POS", "REF", "ALT", "FILTER", "INFO"]] if len(candidates) else candidates

## Siguientes pasos

- Si no aparece nada en los genes candidatos: puede ser build de referencia distinta, o la variante causal está en otro gen de la vía del checkpoint mitótico (ej. `CENPE`, `CENPF`, `NDC80`, `ZWILCH`) — vale la pena ampliar la lista y/o anotar el VCF completo con VEP/SnpEff + ClinVar/gnomAD para priorizar por rareza y predicción de patogenicidad.
- Cruzar los términos del documento de fenotipo (paso 3) contra HPO (Human Phenotype Ontology) para armar una lista de genes candidatos más amplia y priorizada.
- Los FASTQ crudos (85 GB) solo hacen falta si se sospecha que el VCF ya llamado se perdió alguna variante (ej. por filtros del caller) y se quiere re-llamar variantes desde cero.